# Theta functions and Abel-Jacobi inverse via Newton on log Klein theta

This notebook evaluates **Riemann (Klein) theta functions** for a period matrix Ω and implements the **inverse Abel-Jacobi map** using Newton's method on the equation θ(A(z) − u) = 0, with optional use of **log θ** for stability.

## Imports and setup

In [ ]:
import numpy as np
import mpmath as mp

# Optional: run from repo root so the local module is found
try:
    from abel_jacobi_theta import (
        riemann_theta,
        grad_riemann_theta,
        log_theta,
        grad_log_theta,
        inverse_abel_jacobi_newton,
        abel_map_vector,
        omega_vector,
    )
except ImportError:
    import sys
    import os
    # Ensure repo root (where abel_jacobi_theta.py lives) is on path
    sys.path.insert(0, os.getcwd())
    from abel_jacobi_theta import (
        riemann_theta,
        grad_riemann_theta,
        log_theta,
        grad_log_theta,
        inverse_abel_jacobi_newton,
        abel_map_vector,
        omega_vector,
    )

mp.mp.dps = 40

## Hyperelliptic curve: differentials and Abel map (genus 2)

We use the same convention as in `higher_genus_lookup_tables.ipynb`: branch points, ω_k(t) = t^k / √∏(t − a_i), and line integrals from a base point.

In [ ]:
genus = 2
base_point = complex(-3.0, -3.0)

# Branch points for genus 2: 6 points, 3 cuts. Simple symmetric choice.
branch_pts = np.array([
    -1.5 - 0.5j, -1.5 + 0.5j,
    0.0 - 0.8j,  0.0 + 0.8j,
    1.5 - 0.5j,  1.5 + 0.5j,
])

def make_omega(branch_points):
    pts = list(branch_points)
    def omega_k(k, t):
        t = mp.mpc(t) if not isinstance(t, (mp.mpc, mp.mpf)) else t
        prod = mp.mpf(1)
        for a in pts:
            a = mp.mpc(a)
            prod *= (t - a)
        return t**k / mp.sqrt(prod)
    return omega_k

omega = make_omega(branch_pts)

def integrate_omega(k, z):
    z = mp.mpc(z)
    try:
        return complex(mp.quad(lambda t: omega(k, t), [base_point, z]))
    except Exception:
        eps = 1e-12 + 1e-12j
        return complex(mp.quad(lambda t: omega(k, t), [base_point, z + eps]))

def omega_at(z):
    """Return (ω_0(z), ω_1(z)) as numpy array."""
    z = complex(z)
    return np.array([complex(omega(k, z)) for k in range(genus)], dtype=np.complex128)

def A_of_z(z):
    """Abel map from base_point to z."""
    return abel_map_vector(complex(z), integrate_omega, genus)

## Period matrix Ω

For genus 2 we need a 2×2 symmetric matrix with Im(Ω) positive definite. Here we use a **toy matrix** so the theta sum converges quickly. In practice you would compute Ω by integrating the differentials over a- and b-cycles (see comments below).

In [ ]:
# Toy period matrix: symmetric, Im(Ω) >> 0 so theta converges with small N_max
Omega = np.array([
    [1.2 + 2.0j,  0.3 + 0.5j],
    [0.3 + 0.5j,  1.0 + 1.8j],
], dtype=np.complex128)

# Check Im(Ω) is positive definite
assert np.all(np.linalg.eigvalsh(np.imag(Omega)) > 0.1), "Im(Ω) must be positive definite"
print("Ω =")
print(Omega)

## Evaluating the Riemann theta function

θ(z, Ω) = Σ_{n ∈ ℤ^g} exp(πi nᵀΩn + 2πi nᵀz).

In [ ]:
z_test = np.array([0.1 + 0.2j, -0.05 + 0.1j])
th = riemann_theta(z_test, Omega)
print("θ(z_test, Ω) =", th)

grad_th = grad_riemann_theta(z_test, Omega)
print("∇θ(z_test) =", grad_th)

log_th = log_theta(z_test, Omega)
grad_log = grad_log_theta(z_test, Omega)
print("log θ(z_test) =", log_th)
print("∇log θ(z_test) =", grad_log)

## Inverse Abel-Jacobi via Newton on log Klein theta

Given a point **u** in the Jacobian (C^g), find **z** on the curve such that A(z) = u (mod period lattice). We solve θ(A(z) − u) = 0 using Newton, with the option to use **log θ** for stability: G(z) = log θ(A(z)−u), so G'(z) = (∇log θ)·ω(z).

In [ ]:
# Pick a point on the curve and compute its Abel-Jacobi image u = A(z_true)
z_true = 0.5 + 0.3j
u_target = A_of_z(z_true)
print("Target u = A(z_true) for z_true =", z_true)
print("u =", u_target)

# Newton from an initial guess (e.g. perturbed)
z0 = z_true + 0.2 - 0.1j

def A_fun(z):
    return A_of_z(z)

def omega_fun(z):
    return omega_at(z)

z_sol, converged, num_iter = inverse_abel_jacobi_newton(
    u_target,
    Omega,
    A_fun,
    omega_fun,
    z0,
    tol=1e-10,
    max_iter=50,
    use_log_theta=True,
)

print("\nNewton result: z_sol =", z_sol, ", converged =", converged, ", iterations =", num_iter)
print("|z_sol - z_true| =", np.abs(z_sol - z_true))
print("|A(z_sol) - u_target| =", np.linalg.norm(A_of_z(z_sol) - u_target))

## Optional: period matrix from contour integrals (genus 2)

To compute Ω from the curve, integrate ω_k along a- and b-cycles and form the period matrix. Below is a sketch that integrates along simple contours around branch cuts; for production use a proper symplectic basis.

In [ ]:
def contour_integral_omega(omega, k, center, radius, num_pts=64):
    """Integrate ω_k along a circle (center, radius)."""
    def integrand(phi):
        t = center + radius * mp.exp(1j * phi)
        return radius * 1j * mp.exp(1j * phi) * omega(k, t)
    return complex(mp.quad(integrand, [0, 2 * mp.pi], maxdegree=20))

# Example: two contours around first two cuts; columns = periods of (ω_0, ω_1)
# This gives only partial periods; full Ω requires a symplectic basis (a_i, b_i).
c1 = (branch_pts[0] + branch_pts[1]) / 2
r1 = 0.6 * np.abs(branch_pts[0] - branch_pts[1]) / 2
col1 = np.array([contour_integral_omega(omega, k, c1, r1) for k in range(genus)])
c2 = (branch_pts[2] + branch_pts[3]) / 2
r2 = 0.6 * np.abs(branch_pts[2] - branch_pts[3]) / 2
col2 = np.array([contour_integral_omega(omega, k, c2, r2) for k in range(genus)])
periods_approx = np.column_stack([col1, col2])
print("Approximate period columns (two a-type contours):")
print(periods_approx)
print("For full Ω you would compute b-periods and form (I | Ω) from the symplectic basis.")